# Assign Classified Bakery establishments to London MSOAs

This notebook assigns the classified Bakery establishments that we find in the first part of the project to the 2021 MSOA with postcodes as the main method and longitude and latitude used as a backup in case postcodes are missing.

In [20]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

In [21]:
MSOA_PATH = Path("../data/processed/spatial/london_msoa_2021.gpkg")

# Since classification hasn't been completed the entire fhrs dataset is being used as a placeholder for now
FHRS_PATH = Path("../data/interim/business/london_fhrs_prepared_2026-07-23.csv")

POSTCODE_LOOKUP_PATH = Path("../data/raw/spatial_data/msoa_lookup_data.csv")

OUTPUT_PATH = Path("../data/processed/spatial/london_fhrs_with_msoa.csv")

# Checking column names and data structure

Since we will be joining these two datasets, it is important to keep note of the columns of data available and the names that will be required in the final join.

In [22]:
fhrs = pd.read_csv(FHRS_PATH)
london_msoa = gpd.read_file(MSOA_PATH)
postcode_lookup = pd.read_csv(POSTCODE_LOOKUP_PATH, usecols=["pcds", "msoa21cd"], dtype="string")

print(f"FHRS rows: {len(fhrs)}")
print(f"London MSOA rows: {len(london_msoa)}")
print(f"Postcode lookup rows: {len(postcode_lookup)}")

print(f"\nFHRS columns: {fhrs.columns.tolist()}")
print(f"\nLondon MSOA columns: {london_msoa.columns.tolist()}")
print(f"\nPostcode lookup columns: {postcode_lookup.columns.tolist()}")

FHRS rows: 81080
London MSOA rows: 1002
Postcode lookup rows: 2726477

FHRS columns: ['FHRSID', 'BusinessName', 'BusinessType', 'PostCode', 'LocalAuthorityName', 'geocode.longitude', 'geocode.latitude', 'BusinessNameClean']

London MSOA columns: ['borough_name', 'borough_code', 'msoa_name', 'msoa_code', 'area_km2', 'geometry']

Postcode lookup columns: ['pcds', 'msoa21cd']


## Cleaning postcodes

Since MSOA's will be assigned by joining on postcodes, these need to be formatted the same way.

In [23]:
fhrs["postcode_clean"] = (fhrs["PostCode"]
                          .astype("string")
                          .str.upper()
                          .str.replace(" ", "", regex=False)
                          .str.strip()
                          )

postcode_lookup["postcode_clean"] = (postcode_lookup["pcds"]
                                     .astype("string")
                                     .str.upper()
                                     .str.replace(" ", "", regex=False)
                                     .str.strip()
                                     )

fhrs[["PostCode", "postcode_clean"]].sample(10)

,PostCode,postcode_clean
46487,W10 5SY,W105SY
41894,TW7 5NG,TW75NG
20278,UB2 5AA,UB25AA
67430,E1 0RA,E10RA
38529,RM5 3AA,RM53AA
11699,NW1 0AY,NW10AY
76123,SW1P 2QQ,SW1P2QQ
57297,E12 6RH,E126RH
3570,N12,N12
48929,KT1 1JS,KT11JS


# Checking for missing information

This is a quick check so that we can later compare our expectations to the success of the postcode lookups.

In [24]:
postcode_missing = (fhrs["postcode_clean"].isna()| (fhrs["postcode_clean"] == ""))

longitude_missing = pd.to_numeric(fhrs["geocode.longitude"], errors="coerce").isna()
latitude_missing = pd.to_numeric(fhrs["geocode.latitude"], errors="coerce").isna()

geocode_missing = (longitude_missing | latitude_missing)

location_information_missing = (postcode_missing & geocode_missing)

print(f"Total FHRS rows: {len(fhrs)}")
print(f"Missing postcodes: {postcode_missing.sum()}")
print(f"Missing complete geocodes: {geocode_missing.sum()}")
print(f"Neither postcodes nor usable geocodes: {location_information_missing.sum()}")

Total FHRS rows: 81080
Missing postcodes: 1084
Missing complete geocodes: 14433
Neither postcodes nor usable geocodes: 514


# Assigning MSOAs using postcode lookup

After restricting the postcode lookup to that of the 1002 MSOA codes in Greater London, a left join is done onto the Bakery data as the primary assignment method. This will give us business data along with the msoa code that can later be grouped and assigned to a map accordingly.

In [25]:
postcode_lookup_exact = postcode_lookup[["postcode_clean","msoa21cd"]].copy()
postcode_lookup_exact = postcode_lookup_exact[postcode_lookup_exact["msoa21cd"].isin(london_msoa["msoa_code"])]
postcode_lookup_exact = postcode_lookup_exact.drop_duplicates(subset="postcode_clean")
postcode_lookup_exact = postcode_lookup_exact.rename(columns={"msoa21cd": "msoa_code"})

fhrs_msoa = fhrs.merge(
    postcode_lookup_exact,
    on="postcode_clean",
    how="left")

fhrs_msoa["assignment_method"] = pd.NA

fhrs_msoa.loc[fhrs_msoa["msoa_code"].notna(), "assignment_method"] = "postcode"

print(f"Postcode matches: {fhrs_msoa["msoa_code"].notna().sum()}")
print(f"Postcode unmatched: {fhrs_msoa["msoa_code"].isna().sum()}")

fhrs_msoa.sample(10)

Postcode matches: 70371
Postcode unmatched: 10709


,FHRSID,BusinessName,BusinessType,PostCode,LocalAuthorityName,geocode.longitude,geocode.latitude,BusinessNameClean,postcode_clean,msoa_code,assignment_method
33622,422943,Maqsood News,Retailers - other,N4 1AL,Haringey,-0.099738,51.581119,maqsood news,N41AL,E02000419,postcode
74558,1768205,Premier Foods Wholesale,Distributors/Transporters,SW8 5EQ,Wandsworth,NaN,NaN,premier foods wholesale,SW85EQ,E02007083,postcode
56801,805329,Primark,Retailers - other,E6 1HZ,Newham,0.052156,51.535226,primark,E61HZ,E02000731,postcode
11487,424945,Green Note Ltd,Pub/bar/nightclub,NW1 7AN,Camden,-0.145897,51.537265,green note ltd,NW17AN,E02000186,postcode
48255,1958477,IT'S A Date,Other catering premises,W11 1WQ,Kensington and Chelsea,-0.216117,51.513400,it s a date,W111WQ,E02000581,postcode
72213,1845590,"Demi's Deli, Coffee, Salad & Sandwich Bar",Restaurant/Cafe/Canteen,E4 9PT,Waltham Forest,-0.000352,51.607426,demi s deli coffee salad and sandwich bar,E49PT,E02000901,postcode
33041,1833873,Randalls Butchers,Retailers - other,SW6 2TE,Hammersmith and Fulham,-0.191518,51.473302,randalls butchers,SW62TE,E02000394,postcode
58439,1835192,La Pizza & Pasta,Takeaway/sandwich shop,E13 8QE,Newham,0.028617,51.525266,la pizza and pasta,E138QE,E02000739,postcode
11098,423834,Cross Keys,Pub/bar/nightclub,WC2H 9BA,Camden,-0.124885,51.514511,cross keys,WC2H9BA,E02000193,postcode
64384,1291957,Pochi,Takeaway/sandwich shop,SE1 9AH,Southwark,-0.090730,51.505482,pochi,SE19AH,E02000808,postcode


# Checking for unmatched data

This subset of unmatched rows need to be reattempteed for MSOA level assignation using the geocodes.

In [26]:
unmatched_fhrs = fhrs_msoa[fhrs_msoa["msoa_code"].isna()].copy()

unmatched_fhrs["geocode.longitude"] = pd.to_numeric(unmatched_fhrs["geocode.longitude"], errors="coerce")
unmatched_fhrs["geocode.latitude"] = pd.to_numeric(unmatched_fhrs["geocode.latitude"], errors="coerce")

unmatched_fhrs = unmatched_fhrs.dropna(subset=["geocode.longitude", "geocode.latitude"])

print(f"Unmatched rows with usable coordinates: {len(unmatched_fhrs)}")


Unmatched rows with usable coordinates: 644


# Creating fallback for unmatched establishments

Using EPSG:4326 we can convert the spatial points from the geocode to match the London MSOA CRS. The join will assign each point to the MSOA polygon that it is a part of and the successful joins are then added back into the msoa_code column and those that are left unmatched will be assigned as such for future reference.

In [27]:
unmatched_points = gpd.GeoDataFrame(
    unmatched_fhrs,
    geometry=gpd.points_from_xy(unmatched_fhrs["geocode.longitude"],unmatched_fhrs["geocode.latitude"]),
    crs="EPSG:4326"
    ).to_crs(london_msoa.crs)

msoa_for_join = (london_msoa[["msoa_code", "geometry"]].rename(columns={"msoa_code": "coordinate_msoa_code"}))

coordinate_matches = gpd.sjoin(
    unmatched_points,
    msoa_for_join,
    how="inner",
    predicate="within")

fhrs_msoa.loc[coordinate_matches.index, "msoa_code"] = coordinate_matches["coordinate_msoa_code"]

fhrs_msoa.loc[coordinate_matches.index, "assignment_method"] = "coordinates"

fhrs_msoa["assignment_method"] = (fhrs_msoa["assignment_method"].fillna("unmatched"))

# Validation on MSOA lookup of Bakery establishments

A final validation and check is done to understand how successful this method was in assigning bakery establishments to their MSOA codes. Those without MSOA codes assigned may have to be removed or manually inspected and assigned.

In [28]:
final_matches = (fhrs_msoa["msoa_code"].notna().sum())
final_unmatched = (fhrs_msoa["msoa_code"].isna().sum())
final_match_rate = (final_matches / len(fhrs_msoa)* 100)

print(f"Assignment methods: {fhrs_msoa["assignment_method"].value_counts()}")

print(f"\nOriginal FHRS rows: {len(fhrs)}")
print(f"Final FHRS rows: {len(fhrs_msoa)}")
print(f"Final MSOA matches: {final_matches}")
print(f"Final unmatched: {final_unmatched}")
print(f"Final match rate: {final_match_rate:.2f}%")

print(f"Valid MSOA codes: {(fhrs_msoa["msoa_code"].dropna().isin(london_msoa["msoa_code"])).sum()}")

print(f"Duplicate FHRS IDs: {fhrs_msoa["FHRSID"].duplicated().sum()}")

Assignment methods: assignment_method
postcode       70371
unmatched      10079
coordinates      630
Name: count, dtype: int64

Original FHRS rows: 81080
Final FHRS rows: 81080
Final MSOA matches: 71001
Final unmatched: 10079
Final match rate: 87.57%
Valid MSOA codes: 71001
Duplicate FHRS IDs: 0


In [29]:
fhrs_msoa.to_csv(OUTPUT_PATH, index=False)

print(f"Saved to: {OUTPUT_PATH}")

Saved to: ..\data\processed\spatial\london_fhrs_with_msoa.csv
